### Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [64]:
import matplotlib.pyplot as plt
import numpy as np

import os
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Optional
import sys

sys.path.append(os.path.abspath('..'))
import time

import jax
import jax.numpy as jnp
jnp.set_printoptions(precision=None, floatmode='unique')
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_disable_jit", True)
import numpy as np

from floris.floris.floris_model import FlorisModel, TimeSeries

from diffwake.diffwake_jax.turbine.operation_models import power
from diffwake.diffwake_jax.model import load_input, create_state
from diffwake.diffwake_jax import model
from diffwake.diffwake_jax.yaw_runner import make_yaw_runner
from diffwake.diffwake_jax.util import average_velocity_jax
from diffwake.diffwake_jax.turbine.operation_models import power as power_fn
from diffwake.diffwake_jax.simulator import simulate

## Use a 2-turbine debugging layout in JAX to view the calculations

In [78]:
def build_state_runner(
        data_dir: Path,
        farm_yaml: str,
        turbine_yaml: str,
        wind_dir_rad: jax.Array,
        wind_speed: jax.Array,
        turb_intensity: jax.Array,
        dtype,
):
    cfg = load_input(
        str(data_dir / farm_yaml),
        str(data_dir / turbine_yaml),
    ).set(
        wind_directions=wind_dir_rad,
        wind_speeds=wind_speed,
        turbulence_intensities=turb_intensity,
    )
    state = create_state(cfg)
    runner = make_yaw_runner(state)

    x0 = jnp.asarray(cfg.layout["layout_x"], dtype=dtype)
    N = int(x0.shape[0])

    return state, runner, N

DTYPE = jnp.float64
DATA_DIR = Path(rf"data/horn")
FARM_YAML = "gch.yaml"
TURBINE_YAML = "vestas_v802MW.yaml"
WIND_DIR_RAD = jnp.deg2rad(jnp.array([270.0], dtype=DTYPE))
WIND_SPEED = jnp.asarray([9.6], dtype=DTYPE)
TURBULENCE = jnp.full_like(WIND_DIR_RAD, 0.06, dtype=DTYPE)


state, runner, N = build_state_runner(DATA_DIR, FARM_YAML, TURBINE_YAML, WIND_DIR_RAD, WIND_SPEED, TURBULENCE, DTYPE)
yaw_val = jnp.deg2rad(10.0)
yaw_angles = jnp.full((1, N), yaw_val, dtype=DTYPE)
baseline_yaw = jnp.full((1, N), 0.0, dtype=DTYPE)
# print("Yawed run")
out = runner(yaw_angles)


## FLORIS

In [81]:
fmodel = FlorisModel(DATA_DIR / FARM_YAML)
time_series = TimeSeries(wind_speeds=np.array([9.6]),
                   wind_directions=np.array([270.0]),
                   turbulence_intensities=0.06)
fmodel.set(wind_data=time_series)
fyaw = np.full((1, N), 10.0, dtype=np.float64)
fbaseyaw = np.full((1, N), 0.0, dtype=np.float64)

print("FLORIS unyawed run")
fmodel.set(yaw_angles=fbaseyaw)
fmodel.run()
fpower_base = fmodel.get_turbine_powers()

print("\nDiffWake unyawed run")
baseline = runner(baseline_yaw)
print("\nYawed run")
fmodel.set(yaw_angles=fyaw)
fmodel.run()
fpower_yaw = fmodel.get_turbine_powers()
fpower_yaw_expected = fmodel.get_expected_turbine_powers()

FLORIS unyawed run

DiffWake unyawed run

Yawed run


In [89]:
yaw_pow = power(
    state.farm.power_thrust_table,
    out.u_sorted,
    state.flow.air_density,
    yaw_angles=yaw_angles
)
baseline_pow = power(
    state.farm.power_thrust_table,
    baseline.u_sorted,
    state.flow.air_density,
    yaw_angles=baseline_yaw
)
print(f"DiffWake turbine powers (all yawed to {jnp.rad2deg(yaw_val)} degrees): {jnp.sum(yaw_pow)/1e6:.15f}")
print(f"FLORIS turbine powers (all yawed to {fyaw[0,0]} degrees): {np.sum(fpower_yaw)/1e6:.15f}")
print(f"Percentage difference: {abs((np.sum(fpower_yaw) - np.sum(yaw_pow)) / np.sum(fpower_yaw) * 100):.5f}%")

#
print(f"DiffWake turbine powers (unyawed): {jnp.sum(baseline_pow):.15f}")
print(f"FLORIS turbine powers (unyawed): {jnp.sum(fpower_base):.15f}")
print(f"Percentage difference: ")



DiffWake turbine powers (all yawed to 10.0 degrees): 67.274945443995264
FLORIS turbine powers (all yawed to 10.0 degrees): 67.028446546648482
Percentage difference: 0.36775%
DiffWake turbine powers (unyawed): 63505928.401255056262016
FLORIS turbine powers (unyawed): 63505928.401255071163177
Percentage difference: 


### Results
Clearly from the unyawed case, waked turbines are handled differently

In [68]:
# DiffWake final velocity grid
diffwake_vel = baseline.u_sorted
diffwake_avg_vel = average_velocity_jax(diffwake_vel)
diffwake_vel

Array([[[[9.220105039547377, 9.6              , 9.893923985303262],
         [9.220105039547377, 9.6              , 9.893923985303262],
         [9.220105039547377, 9.6              , 9.893923985303262]],

        [[7.215913527536866, 7.153441553942855, 7.743263186237632],
         [6.87037570013198 , 6.731636309781524, 7.372472942121433],
         [7.215935597262278, 7.153468494917699, 7.743286868851298]]]],      dtype=float64)

In [69]:
fmodel.core.flow_field.u_sorted

array([[[[9.220105039547377, 9.6              , 9.893923985303262],
         [9.220105039547377, 9.6              , 9.893923985303262],
         [9.220105039547377, 9.6              , 9.893923985303262]],

        [[7.215913527536866, 7.153441553942855, 7.743263186237632],
         [6.87037570013198 , 6.731636309781524, 7.372472942121433],
         [7.215935597262278, 7.153468494917699, 7.743286868851298]]]])